# Catch–Slip Adhesion Model with Physics-Informed Neural Networks

This notebook solves the time-dependent **catch–slip bond adhesion** model using Physics-Informed Neural Networks (PINNs). The model describes the coupled dynamics of bond density $\phi(t)$ and bond extension $u(t)$:

$$
\frac{d\phi}{dt} = k_{on}\,(\phi_m - \phi) - k_{off}(F)\,\phi
$$

$$
\frac{du}{dt} = v_0 - k_{off}(F)\,u
$$

where the catch–slip off-rate depends on force $F = \kappa u$ via:

$$
k_{off}(F) = k_c\,e^{-F/F_c} + k_s\,e^{F/F_s}
$$

The two terms model competing **catch** (force-strengthened, $k_c$) and **slip** (force-weakened, $k_s$) unbinding pathways.

We demonstrate two PINN approaches:
1. **Standard PINN**: A single network trained on the full time domain
2. **FBPINN**: Domain decomposition with multiple smaller networks

In [ ]:
import pinns
import torch
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.interpolate import interp1d

# Define the Problem

## Parameters of the Model

Physical parameters for the catch–slip system:
- **$k_{on}$**: Binding rate constant
- **$\phi_m$**: Maximum bond density (normalised)
- **$k_c$, $F_c$**: Catch-bond rate constant and force scale
- **$k_s$, $F_s$**: Slip-bond rate constant and force scale
- **$\kappa$**: Spring stiffness (force per unit extension)
- **$v_0$**: Prescribed pulling velocity
- **$\phi_0$, $u_0$**: Initial conditions at $t=0$

In [ ]:
k_on  = 1.0    # binding rate
phi_m = 1.0    # maximum bond density
k_c   = 1.0    # catch off-rate constant
F_c   = 10.0   # catch force scale
k_s   = 0.5    # slip off-rate constant
F_s   = 5.0    # slip force scale
kappa = 1.0    # spring stiffness
v_0   = 2.0    # pulling velocity

phi0  = 0.0    # initial bond density
u0    = 0.0    # initial extension
t_max = 20.0   # simulation time

## Domain

The problem is purely temporal, so the domain is the 1D interval $[0, t_{max}]$.

In [ ]:
domain = pinns.DomainCubic(
    [0.0],
    [t_max],
)

## Differential Equations

The `model_catchslip` function returns the residuals of the two coupled ODEs.

Substituting $F = \kappa u$, the reduced system is:

$$
\mathcal{R}_\phi = \frac{d\phi}{dt} - k_{on}(\phi_m - \phi) + \left(k_c e^{-\kappa u/F_c} + k_s e^{\kappa u/F_s}\right)\phi = 0
$$

$$
\mathcal{R}_u = \frac{du}{dt} - v_0 + \left(k_c e^{-\kappa u/F_c} + k_s e^{\kappa u/F_s}\right)u = 0
$$

**Column convention**
- `X`: $[t]$ — shape $(N, 1)$
- `U`: $[\phi, u]$ — shape $(N, 2)$

In [ ]:
def model_catchslip(X, U, params):
    k_on  = params["fixed"]["k_on"]
    phi_m = params["fixed"]["phi_m"]
    k_c   = params["fixed"]["k_c"]
    F_c   = params["fixed"]["F_c"]
    k_s   = params["fixed"]["k_s"]
    F_s   = params["fixed"]["F_s"]
    kappa = params["fixed"]["kappa"]
    v_0   = params["fixed"]["v_0"]

    phi = U[:, 0:1]
    u   = U[:, 1:2]

    # Force–extension relation
    F = kappa * u

    # Catch–slip off-rate
    k_off = k_c * torch.exp(-F / F_c) + k_s * torch.exp(F / F_s)

    # Time derivatives
    phi_t = pinns.derivative(phi, X, 0, (0,))
    u_t   = pinns.derivative(u,   X, 0, (0,))

    # Residuals
    res_phi = phi_t - k_on * (phi_m - phi) + k_off * phi
    res_u   = u_t   - v_0                  + k_off * u

    return (res_phi, res_u)

## Initial Conditions

At $t = 0$ both the bond density and extension start from zero:

$$\phi(0) = 0, \qquad u(0) = 0$$

In [ ]:
domain.add_dirichlet(
    boundary=(0,),
    value=phi0,
    component=0,
    name="initial_phi"
)
domain.add_dirichlet(
    boundary=(0,),
    value=u0,
    component=1,
    name="initial_u"
)

## Reference Solution

We integrate the ODE numerically with `scipy` to obtain a reference solution for validation.

In [ ]:
def model_catchslip_reference(X, params):
    p = params["fixed"]

    def ode(t, y):
        phi, u = y
        F     = p["kappa"] * u
        k_off = p["k_c"] * np.exp(-F / p["F_c"]) + p["k_s"] * np.exp(F / p["F_s"])
        return [
            p["k_on"] * (p["phi_m"] - phi) - k_off * phi,
            p["v_0"] - k_off * u,
        ]

    t = X[:, 0].flatten()
    t_dense = np.linspace(0.0, t.max(), 2000)
    sol = solve_ivp(
        ode,
        [0.0, t.max()],
        [p["phi0"], p["u0"]],
        t_eval=t_dense,
        method="RK45",
        rtol=1e-9,
        atol=1e-11,
    )

    phi_ref = interp1d(sol.t, sol.y[0], kind="cubic")(t)
    u_ref   = interp1d(sol.t, sol.y[1], kind="cubic")(t)

    return np.column_stack([phi_ref, u_ref])

## Assemble the Problem

We combine the domain, residual function, parameters, and reference solution into a `Problem` object.

The `output_range` specifies the expected physical range of each output, used to unnormalize the network predictions:
- $\phi \in [0,\, \phi_m]$
- $u \in [0,\, u_{max}]$ where $u_{max}$ is estimated from the steady-state value $u^* = v_0 / k_{off}(F^*)$

In [ ]:
# Estimate steady-state extension (iterative)
u_ss = v_0 / (k_c + k_s)  # zero-force estimate
for _ in range(20):
    F_ss  = kappa * u_ss
    koff  = k_c * np.exp(-F_ss / F_c) + k_s * np.exp(F_ss / F_s)
    u_ss  = v_0 / koff
u_max = 2.0 * u_ss  # add margin
print(f"Estimated steady-state extension u* = {u_ss:.3f}  (u_max = {u_max:.3f})")

problem = pinns.Problem(
    domain,
    model_catchslip,
    input_names=["t"],
    output_names=["phi", "u"],
    params={
        "k_on":  k_on,
        "phi_m": phi_m,
        "k_c":   k_c,
        "F_c":   F_c,
        "k_s":   k_s,
        "F_s":   F_s,
        "kappa": kappa,
        "v_0":   v_0,
        "phi0":  phi0,
        "u0":    u0,
    },
    output_range=[(0.0, phi_m), (0.0, u_max)],
    solution=model_catchslip_reference,
)

# Solve with Standard PINN

## Network Architecture

A fully-connected network with two hidden layers and `tanh` activations. The network takes $t$ as input and outputs $(\phi, u)$.

In [ ]:
network = pinns.FNN(
    [1, 64, 64, 64, 2],
    activation="tanh",
    normalize_input=True,
    unnormalize_output=True,
)

## Training

We use Adam with 30 000 epochs. The initial conditions are sampled at a single point ($t = 0$) with higher weight to ensure they are exactly satisfied.

In [ ]:
trainer = pinns.Trainer(problem, network)

trainer.compile(
    train_samples={
        "pde":         1000,
        "initial_phi": 1,
        "initial_u":   1,
    },
    test_samples={
        "pde":         1000,
        "initial_phi": 0,
        "initial_u":   0,
    },
    weights={
        "pde":         1.0,
        "initial_phi": 10.0,
        "initial_u":   10.0,
    },
    optimizer="adam",
    learning_rate=1e-3,
    epochs=30000,
    print_each=1000,
    show_plots=True,
)

trainer.train()

# Solve with FBPINN (Finite Basis PINN)

For longer time horizons the solution can develop multiple time-scales (fast initial transient + slow approach to steady state). FBPINN decomposes the interval into overlapping subdomains, each handled by a smaller network, which improves accuracy at lower computational cost.

## Domain Partition

In [ ]:
domain_partition = pinns.DomainCubicPartition(
    [np.linspace(0.0, t_max, 12)],
    overlap=0.5,
)

domain_partition.add_dirichlet(
    boundary=(0,),
    value=phi0,
    component=0,
    name="initial_phi"
)
domain_partition.add_dirichlet(
    boundary=(0,),
    value=u0,
    component=1,
    name="initial_u"
)

In [ ]:
problem_fb = pinns.Problem(
    domain_partition,
    model_catchslip,
    input_names=["t"],
    output_names=["phi", "u"],
    params={
        "k_on":  k_on,
        "phi_m": phi_m,
        "k_c":   k_c,
        "F_c":   F_c,
        "k_s":   k_s,
        "F_s":   F_s,
        "kappa": kappa,
        "v_0":   v_0,
        "phi0":  phi0,
        "u0":    u0,
    },
    output_range=[(0.0, phi_m), (0.0, u_max)],
    solution=model_catchslip_reference,
)

## FBPINN Network and Training

In [ ]:
base_network = pinns.FNN([1, 32, 32, 2], activation="tanh")

network_fb = pinns.FBPINN(
    domain_partition,
    base_network,
    normalize_input=True,
    unnormalize_output=True,
)

trainer_fb = pinns.Trainer(problem_fb, network_fb)

trainer_fb.compile(
    train_samples={
        "pde":         1000,
        "initial_phi": 1,
        "initial_u":   1,
    },
    test_samples={
        "pde":         1000,
        "initial_phi": 0,
        "initial_u":   0,
    },
    weights={
        "pde":         1.0,
        "initial_phi": 10.0,
        "initial_u":   10.0,
    },
    optimizer="adam",
    learning_rate=1e-3,
    epochs=15000,
    print_each=500,
    show_plots=True,
)

trainer_fb.train()

# Steady-State Analysis

At steady state $d\phi/dt = 0$ and $du/dt = 0$, giving:

$$
u^* = \frac{v_0}{k_{off}(F^*)}, \qquad F^* = \kappa u^*, \qquad
\phi^* = \frac{k_{on}\,\phi_m}{k_{on} + k_{off}(F^*)}
$$

We verify that the PINN prediction converges to these values.

In [ ]:
# Steady-state values (already converged above)
F_ss   = kappa * u_ss
koff_ss = k_c * np.exp(-F_ss / F_c) + k_s * np.exp(F_ss / F_s)
phi_ss = k_on * phi_m / (k_on + koff_ss)
print(f"Steady-state:  u* = {u_ss:.4f},  F* = {F_ss:.4f},  phi* = {phi_ss:.4f}")

# Evaluate PINN at t = t_max
t_end = np.array([[t_max]])
pred  = trainer.predict(t_end)[0]
print(f"PINN at t={t_max}: phi = {pred[0]:.4f} (ref {phi_ss:.4f}), "
      f"u = {pred[1]:.4f} (ref {u_ss:.4f})")

# Final Comparison Plot

Compare the PINN solution, FBPINN solution, and the scipy reference over the full time interval.

In [ ]:
t_plot  = np.linspace(0.0, t_max, 500).reshape(-1, 1)
params_ref = trainer._build_params()

y_ref  = model_catchslip_reference(t_plot, params_ref)
y_pred = trainer.predict(t_plot)
y_fb   = trainer_fb.predict(t_plot)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

labels   = [r"$\phi(t)$ — bond density", r"$u(t)$ — extension"]
ss_vals  = [phi_ss, u_ss]

for i, (ax, label, ss) in enumerate(zip(axes, labels, ss_vals)):
    ax.plot(t_plot, y_ref[:, i],  'k--',  linewidth=2,   label='Reference (scipy)')
    ax.plot(t_plot, y_pred[:, i], 'b-',   linewidth=1.5, label='Standard PINN')
    ax.plot(t_plot, y_fb[:, i],   'r-',   linewidth=1.5, label='FBPINN')
    ax.axhline(ss, color='gray', linestyle=':', linewidth=1, label=f'Steady state = {ss:.3f}')
    ax.set_xlabel('$t$')
    ax.set_title(label)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Catch–Slip Adhesion Model', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()